## Demo — Deriving and Computing One KRI in Python

**Scenario.** A B2B SaaS company runs **Aurora Assistant**, an LLM-powered customer-support assistant in production. The risk register has one row that drives this KRI:

> **R-03**: *Anomalous refusal-rate spikes on the assistant indicate either policy drift or attempted misuse — both warrant investigation.*

We will derive the KRI live: **refusal rate** (% of turns the assistant refuses to answer), set green / amber / red thresholds, implement it as a small pandas function, and emit the standard KRI return shape `{value, status, owner, timestamp}`.

> **Capstone through-line.** That return shape is the same contract as the capstone project's `lab/kri.py`. If you write the function with this shape today, you re-use it on the project — no relearning.

## 1. Load the 30-day monitoring log

In [1]:
import pandas as pd
from datetime import datetime, timezone
from pathlib import Path

DATA = Path('data/refusal_log.csv')
df = pd.read_csv(DATA, parse_dates=['timestamp'])
print(f'Rows (turns): {len(df):,}')
print(f'Sessions:     {df.session_id.nunique():,}')
print(f'Days covered: {df.timestamp.dt.date.nunique()}')
df.head()

Rows (turns): 6,668
Sessions:     1,915
Days covered: 30


,timestamp,session_id,model_version,turn_index,refusal_flag
0,2026-04-08 00:06:25+00:00,S100005,aurora-2.3,1,0
1,2026-04-08 00:08:49+00:00,S100002,aurora-2.3,2,0
2,2026-04-08 00:19:46+00:00,S100033,aurora-2.3,1,0
3,2026-04-08 00:26:47+00:00,S100060,aurora-2.3,5,0
4,2026-04-08 00:41:47+00:00,S100042,aurora-2.3,1,0


## 2. Define the threshold band

Per the KRI definition (single risk-register row → single signal → single threshold):

| Band  | Refusal rate (rolling 24h) | Meaning |
|-------|----------------------------|---------|
| Green | ≤ 3%                       | Within normal variance — guardrails working as designed |
| Amber | 3% – 7%                    | Investigate: probable policy drift or content-mix shift |
| Red   | > 7%                       | Escalate: likely misuse campaign or serious policy regression |

## 3. Compute the rolling refusal rate

In [2]:
def status_band(value, green=0.03, red=0.07):
    if value <= green: return 'green'
    if value <= red:   return 'amber'
    return 'red'

# Daily refusal rate over the trailing 30 days
by_day = (df.set_index('timestamp')
            .groupby(pd.Grouper(freq='D'))['refusal_flag']
            .mean())
print(by_day.tail(10).to_string())

timestamp
2026-04-28 00:00:00+00:00    0.020661
2026-04-29 00:00:00+00:00    0.113990
2026-04-30 00:00:00+00:00    0.064677
2026-05-01 00:00:00+00:00    0.013575
2026-05-02 00:00:00+00:00    0.040161
2026-05-03 00:00:00+00:00    0.012931
2026-05-04 00:00:00+00:00    0.056995
2026-05-05 00:00:00+00:00    0.033088
2026-05-06 00:00:00+00:00    0.021164
2026-05-07 00:00:00+00:00    0.025000
Freq: D


## 4. Wrap in the standard KRI return shape

In [3]:
def kri_refusal_rate(df, green=0.03, red=0.07,
                     owner='LLM Platform Lead'):
    """
    Daily refusal rate on Aurora Assistant.

    Returns the standard KRI shape: {value, status, owner, timestamp}.
    Same contract as the capstone's lab/kri.py.
    """
    value = float(df['refusal_flag'].mean())
    return {
        'value': round(value, 4),
        'status': status_band(value, green=green, red=red),
        'owner': owner,
        'timestamp': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    }

# Apply to the most recent 24h window
latest_day = df['timestamp'].dt.date.max()
today_df = df[df['timestamp'].dt.date == latest_day]
result = kri_refusal_rate(today_df)
result

{'value': 0.025,
 'status': 'green',
 'owner': 'LLM Platform Lead',
 'timestamp': '2026-05-08T19:03:55+00:00'}

## 5. Spot-check the spike days

There should be at least one **red** day in the trailing 30 — that's the moment the dashboard would have paged the LLM Platform Lead, and the moment the AI review board would have asked _"is this policy drift or are we under attack?"_

In [4]:
rates = by_day.dropna().sort_values(ascending=False).head(5)
for day, rate in rates.items():
    print(f'{day.date()}  refusal rate = {rate:.3%}  → {status_band(rate)}')

2026-04-29  refusal rate = 11.399%  → red
2026-04-30  refusal rate = 6.468%  → amber
2026-05-04  refusal rate = 5.699%  → amber
2026-04-21  refusal rate = 4.783%  → amber
2026-05-02  refusal rate = 4.016%  → amber


## Key takeaway

A KRI is not a dashboard widget. It is a **risk signal with a threshold, an escalation path, a named owner, and a small Python function that produces the value reproducibly** — the dashboard is just where the function's output shows up.

The exercise scales this pattern to a 5-KRI portfolio. The two coded KRIs (subgroup FNR delta + drift score) reuse the exact same `{value, status, owner, timestamp}` shape — the same contract the capstone's `lab/kri.py` expects.